# Notebook 01 — Data Audit

Mục tiêu: kiểm tra **dữ liệu gốc trước khi làm sạch**: số dòng, missing, dòng trùng, timestamp trùng, timestamp không hợp lệ và giờ bị thiếu. Không loại bỏ các dòng missing trước khi audit.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

# Tìm gốc repo có chứa file dữ liệu gốc để đường dẫn chạy được trên Colab.
DATA_RELATIVE = Path("PRSA_data_2010.1.1-2014.12.31.csv")
repo_candidates = [
    Path.cwd(),
    Path("/content/PPNCKH_5Bros"),
    Path("/content"),
]
PROJECT_ROOT = next(
    (root.resolve() for root in repo_candidates if (root / DATA_RELATIVE).exists()),
    None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Không tìm thấy dữ liệu gốc: {DATA_RELATIVE}. "
        "Kiểm tra repo đã được clone vào /content/PPNCKH_5Bros và file raw nằm đúng vị trí."
    )

DATA_PATH = PROJECT_ROOT / DATA_RELATIVE
OUTPUT_ROOT = PROJECT_ROOT / "PPNCKH_5Bros/notebooks/phuong_nb/output"
DATA_OUT = OUTPUT_ROOT / "data"
FIG_OUT = OUTPUT_ROOT / "figures"
TABLE_OUT = OUTPUT_ROOT / "tables"
for folder in (DATA_OUT, FIG_OUT, TABLE_OUT):
    folder.mkdir(parents=True, exist_ok=True)

def load_raw_data():
    data = pd.read_csv(DATA_PATH)
    required = {"year", "month", "day", "hour"}
    missing_cols = required - set(data.columns)
    if missing_cols:
        raise ValueError(f"Thiếu cột thời gian: {sorted(missing_cols)}")
    data["datetime"] = pd.to_datetime(data[["year", "month", "day", "hour"]], errors="coerce")
    return data

print("Repo root:", PROJECT_ROOT)
print("Data file:", DATA_PATH)
print("Output root:", OUTPUT_ROOT)

Repo root: /content
Data file: /content/PRSA_data_2010.1.1-2014.12.31.csv
Output root: /content/PPNCKH_5Bros/notebooks/phuong_nb/output


In [2]:
df_raw = load_raw_data()

# Audit giữ nguyên toàn bộ các dòng được đọc từ CSV.
n_rows = len(df_raw)
n_duplicate_rows = int(df_raw.duplicated().sum())
n_invalid_timestamp = int(df_raw["datetime"].isna().sum())
n_duplicate_timestamps = int(df_raw.loc[df_raw["datetime"].notna(), "datetime"].duplicated().sum())

missing_summary = (
    df_raw.isna().sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_percent=lambda x: (x["missing_count"] / max(n_rows, 1) * 100).round(2))
    .reset_index(names="column")
)

valid_times = df_raw.loc[df_raw["datetime"].notna(), "datetime"].sort_values()
if not valid_times.empty:
    full_hourly_index = pd.date_range(valid_times.min(), valid_times.max(), freq="h")
    observed_hours = pd.DatetimeIndex(valid_times.unique())
    missing_hours = full_hourly_index.difference(observed_hours)
else:
    full_hourly_index = pd.DatetimeIndex([])
    missing_hours = pd.DatetimeIndex([])

audit_summary = pd.DataFrame([
    {"metric": "total_rows_in_raw_csv", "value": n_rows},
    {"metric": "duplicate_entire_rows", "value": n_duplicate_rows},
    {"metric": "invalid_timestamps", "value": n_invalid_timestamp},
    {"metric": "duplicate_timestamps", "value": n_duplicate_timestamps},
    {"metric": "unique_valid_timestamps", "value": int(valid_times.nunique())},
    {"metric": "expected_hourly_timestamps_in_range", "value": len(full_hourly_index)},
    {"metric": "missing_hour_timestamps", "value": len(missing_hours)},
    {"metric": "missing_pm2.5_count", "value": int(df_raw["pm2.5"].isna().sum()) if "pm2.5" in df_raw else np.nan},
])

display(audit_summary)
display(missing_summary)

audit_summary.to_csv(TABLE_OUT / "beijing_data_quality_report.csv", index=False)
missing_summary.to_csv(TABLE_OUT / "beijing_missing_by_column.csv", index=False)
pd.DataFrame({"missing_datetime": missing_hours}).to_csv(TABLE_OUT / "beijing_missing_timestamps.csv", index=False)

print("Đã lưu báo cáo audit vào:", TABLE_OUT)

,metric,value
0,total_rows_in_raw_csv,43824
1,duplicate_entire_rows,0
2,invalid_timestamps,0
3,duplicate_timestamps,0
4,unique_valid_timestamps,43824
5,expected_hourly_timestamps_in_range,43824
6,missing_hour_timestamps,0
7,missing_pm2.5_count,2067


,column,missing_count,missing_percent
0,No,0,0.00
1,year,0,0.00
2,month,0,0.00
3,day,0,0.00
4,hour,0,0.00
5,pm2.5,2067,4.72
6,DEWP,0,0.00
7,TEMP,0,0.00
8,PRES,0,0.00
9,cbwd,0,0.00


Đã lưu báo cáo audit vào: /content/PPNCKH_5Bros/notebooks/phuong_nb/output/tables


**Lưu ý:** timestamp bị thiếu được xác định bằng cách so sánh các timestamp hợp lệ duy nhất với dải giờ liên tục từ thời điểm đầu đến cuối. Missing trong cột PM2.5 được báo cáo riêng, không tự động xóa hoặc nội suy.